## Create Catalog and Schema 

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS my_first_catalog;

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS my_first_catalog.my_first_schema;

In [0]:
%sql
USE CATALOG my_first_catalog;
SHOW SCHEMAS;

In [0]:
from delta.tables import *
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, array, ArrayType, DateType, TimestampType, FloatType
from pyspark.sql.functions import *

## Create Delta Table using DataFrame

In [0]:
df_sales_orders = spark.read                           \
                         .option("header", "true")      \
                         .option("inferSchema", "true") \
                         .csv("/Volumes/my_first_catalog/my_first_schema/my_staging_data/Data/store_orders_H1.csv")
df_sales_orders = df_sales_orders.withColumn("order_date", to_date(df_sales_orders.order_date, 'MM/dd/yyyy'))
display(df_sales_orders)
df_sales_orders.printSchema()

In [0]:
%sql
DROP TABLE IF EXISTS my_first_catalog.my_first_schema.store_orders;

### Create Delta Table with Partitioning 

In [0]:
df_sales_orders.write.format("delta").partitionBy("currency").saveAsTable("my_first_catalog.my_first_schema.store_orders")

### Create Delta Table with Liquid Clustering

In [0]:
%sql
DROP TABLE IF EXISTS my_first_catalog.my_first_schema.store_orders_clusters

In [0]:
df_sales_orders.write \
  .format("delta") \
  .clusterBy("order_number") \
  .saveAsTable("my_first_catalog.my_first_schema.store_orders_clusters")

In [0]:
%sql
SHOW TBLPROPERTIES my_first_catalog.my_first_schema.store_orders_clusters;

In [0]:
%%sql		
SELECT * FROM my_first_catalog.my_first_schema.store_orders;

### Enable AutoOptimize for optimizeWrite and optimizeCompact 

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders 
SET TBLPROPERTIES (
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)

In [0]:
%%sql
DESCRIBE DETAIL my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
SHOW TBLPROPERTIES my_first_catalog.my_first_schema.store_orders;

In [0]:
%%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

## Using replaceWhere feature of Databricks

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.store_orders WHERE order_date BETWEEN '2026-01-01' AND '2026-01-31';

In [0]:
replace_data = (
    spark.table("my_first_catalog.my_first_schema.store_orders")                             
         .filter("order_date BETWEEN '2026-01-01' AND '2026-01-31'")  
         .withColumn("units", col("units") + 2)       
         .withColumn("updated_at", current_timestamp()) 
)


In [0]:
(replace_data.write
  .mode("overwrite")
  .option("replaceWhere", "order_date BETWEEN '2026-01-01' AND '2026-01-31'")
  .saveAsTable("my_first_catalog.my_first_schema.store_orders")
)

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.store_orders WHERE order_date BETWEEN '2026-01-01' AND '2026-01-31';


In [0]:
%sql
DESCRIBE DETAIL my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.store_orders;

## Adding new column in Delta Table

### Adding column at a specific position 

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders ADD COLUMNS (dummy_col STRING AFTER order_number);

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.store_orders;


In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

### Adding column at the first position

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders ALTER COLUMN dummy_col FIRST;

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

## Renaming a column in Delta Table

### Make sure to enable column mapping to rename the column

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders RENAME COLUMN dummy_col TO new_dummy_col;

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders SET TBLPROPERTIES ('delta.columnMapping.mode' ='name')

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders RENAME COLUMN dummy_col TO new_dummy_col;

In [0]:
%sql
DESCRIBE TABLE my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
SHOW TBLPROPERTIES my_first_catalog.my_first_schema.store_orders;

## Drop the column using DROP COLUMN command

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders DROP COLUMN new_dummy_col;

In [0]:
%sql
DESCRIBE TABLE my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

## Setting up Constraints on Delta Table

## NULL constraint is an enforced constraint which fails the transaction when violated

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders
ALTER COLUMN order_number SET NOT NULL;

In [0]:
%sql
INSERT INTO my_first_catalog.my_first_schema.store_orders (order_number,customer_id,product_id,order_date,units,sale_price,currency,order_mode) VALUES (NULL,101,1,'2026-01-01',1,100,'USD','NEW');
    

### CHECK constraint is also an enforced constraint which enforces a condition on column values so that only rows satisfying that condition can be inserted or updated in the table.

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders ADD CONSTRAINT valid_date CHECK (order_date >= '2025-01-01' and order_date < '2027-01-01');
DESCRIBE DETAIL my_first_catalog.my_first_schema.store_orders;
SHOW TBLPROPERTIES my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
INSERT INTO my_first_catalog.my_first_schema.store_orders (order_number,customer_id,product_id,order_date,units,sale_price,currency,order_mode) VALUES (2511,101,1,'2027-01-01',1,100,'USD','NEW');

## Create a Delta Table to showcase the Foreign Key and Primary Key constraints in Databricks

### Foreign Key and Primary Key constraints in Databricks are informational and are not enforced

In [0]:
%sql
DROP TABLE IF EXISTS my_first_catalog.my_first_schema.store_customers

In [0]:
%sql
CREATE TABLE IF NOT EXISTS my_first_catalog.my_first_schema.store_customers (customer_id INT PRIMARY KEY, customer_name VARCHAR(255), address VARCHAR(255), city VARCHAR(255), country VARCHAR(100), phone VARCHAR(100), email VARCHAR(255), credit_card VARCHAR(255));

INSERT INTO my_first_catalog.my_first_schema.store_customers 
VALUES
(1,'Chanchal Ismail','Ap #238-278 Nulla Road','Silvassa','India','+91 5969847806','non.quam.Pellentesque@auctorveliteget.co.uk','5396329326153941'),(3,'Remedios Kline','P.O. Box 658, 5723 Pede Street','Rimbey','Canada','1 (587) 590-6469','lobortis.ultrices@eusem.org','5373106780298207'),(7,'Ira Lucas','936-3011 Convallis Road','Shreveport','United States','1-117-676-2784','nec@lectus.net','5164946381862809'),(5,'Blair Bowen','Ap #111-9306 Cum Ave','Racine','United States','1-831-726-0593','neque@Donecegestas.com','5568171323437295')

In [0]:
%sql
DESCRIBE EXTENDED my_first_catalog.my_first_schema.store_customers;

### Adding Foreign Key constraint in the existing store_orders Delta Table with reference to the store_customers Dellta Table created

### Ideally in SQL databases the following command would have failed because all the records present in store

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders 
ADD FOREIGN KEY (customer_id) REFERENCES my_first_catalog.my_first_schema.store_customers(customer_id);

In [0]:
%sql
DESCRIBE EXTENDED my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
SELECT *
FROM my_first_catalog.my_first_schema.store_orders o
JOIN my_first_catalog.my_first_schema.store_customers c
  ON o.customer_id = c.customer_id;

In [0]:
%sql
INSERT INTO my_first_catalog.my_first_schema.store_customers (customer_id,customer_name,address,city,country,phone,email,credit_card) VALUES (1,'John Doe','123 Main Street','New York','USA','516-276-0548','johndoe@example.com','5469770617978526')

In [0]:
%sql
EXPLAIN SELECT *
FROM my_first_catalog.my_first_schema.store_orders o
JOIN my_first_catalog.my_first_schema.store_customers c
  ON o.customer_id = c.customer_id;

In [0]:
%sql
DROP TABLE my_first_catalog.my_first_schema.store_customers;

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders
DROP CONSTRAINT IF EXISTS store_orders_store_customers_fk;

In [0]:
SHOW PRIMARY KEY/ FOREIGN KEY RELATION WITH RELY, QUERY OPTIMIZATION USING JOIN  

In [0]:
spark.conf.set(
    "spark.databricks.delta.commitInfo.userMetadata",
    "Generated unit_price column"
)


In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders
ADD COLUMN unit_price DATE
GENERATED ALWAYS AS (
  CAST(sale_price AS INT) 
);


In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders
ADD COLUMN eventDate DATE GENERATED ALWAYS AS (CAST(eventTime AS DATE));

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders
ADD COLUMNS (
    name_length INT GENERATED ALWAYS AS (length(order_number))
);

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.store_orders WHERE order_number=5; 


In [0]:
%sql
UPDATE my_first_catalog.my_first_schema.store_orders SET sale_price=90.50 WHERE order_number=5;

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.store_orders WHERE order_number=5;

In [0]:
%%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.store_orders VERSION AS OF 0 WHERE order_number=5;

In [0]:
%sql
DELETE FROM my_first_catalog.my_first_schema.store_orders WHERE order_number=5;
SELECT * FROM my_first_catalog.my_first_schema.store_orders WHERE order_number=5;

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
RESTORE TABLE my_first_catalog.my_first_schema.store_orders TO VERSION AS OF 10;
SELECT * FROM my_first_catalog.my_first_schema.store_orders VERSION AS OF 9 WHERE order_number=5; 

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
SELECT count(*) FROM my_first_catalog.my_first_schema.store_orders;

In [0]:
df_store_orders_CDC1 = spark.read                          \
                         .option("header", "true")      \
                         .option("inferSchema", "true") \
                         .csv("/Volumes/my_first_catalog/my_first_schema/my_staging_data/Data/store_orders_H2.csv")
df_store_orders_CDC1 = df_store_orders_CDC1.withColumn("order_date", to_date(df_store_orders_CDC1.order_date, 'MM/dd/yyyy'))
display(df_store_orders_CDC1)
df_store_orders_CDC1.printSchema()
df_store_orders_CDC1.createOrReplaceTempView("cdc_store_orders_1")


In [0]:
spark.sql("SELECT * FROM cdc_store_orders_1").show()

In [0]:
%sql
MERGE WITH SCHEMA EVOLUTION INTO my_first_catalog.my_first_schema.store_orders
USING cdc_store_orders_1
ON cdc_store_orders_1.order_number = my_first_catalog.my_first_schema.store_orders.order_number
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *

In [0]:
%sql
SELECT count(*) FROM my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
DESCRIBE DETAIL my_first_catalog.my_first_schema.store_orders

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.store_orders;

In [0]:
df_store_orders_CDC2 = spark.read                          \
                         .option("header", "true")      \
                         .option("inferSchema", "true") \
                         .csv("/Volumes/my_first_catalog/my_first_schema/my_staging_data/Data/store_orders_H3.csv")

df_store_orders_CDC2 = df_store_orders_CDC2.withColumn("order_date",to_date(df_store_orders_CDC2.order_date, "MM/dd/yyyy"))

display(df_store_orders_CDC2)
df_store_orders_CDC2.printSchema()
df_store_orders_CDC2.createOrReplaceTempView("cdc_store_orders_2")

In [0]:
%sql
MERGE WITH SCHEMA EVOLUTION INTO my_first_catalog.my_first_schema.store_orders
USING cdc_store_orders_2
ON cdc_store_orders_2.order_number = my_first_catalog.my_first_schema.store_orders.order_number
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.store_orders WHERE order_number IN (500,1254, 1501, 2234, 2345);

In [0]:
%sql
SELECT count(*) FROM my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

In [0]:
df_sales_orders_CDC3 = spark.read                          \
                         .option("header", "true")      \
                         .option("inferSchema", "true") \
                         .csv("/Volumes/my_first_catalog/my_first_schema/my_staging_data/Data/store_orders_H4.csv")

df_sales_orders_CDC3 = df_sales_orders_CDC3.withColumn("order_date", to_date(df_sales_orders_CDC3.order_date, "MM/dd/yyyy"))

display(df_sales_orders_CDC3)
df_sales_orders_CDC3.printSchema()
df_sales_orders_CDC3.createOrReplaceTempView("cdc_store_orders_3")

In [0]:
%sql
MERGE WITH SCHEMA EVOLUTION INTO my_first_catalog.my_first_schema.store_orders
USING cdc_store_orders_3
ON cdc_store_orders_3.order_number = my_first_catalog.my_first_schema.store_orders.order_number
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *

In [0]:
%sql
SELECT * FROM my_first_catalog.my_first_schema.store_orders WHERE order_number IN (1501, 2345);

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
DESCRIBE DETAIL my_first_catalog.my_first_schema.store_orders;


In [0]:
%sql
SELECT version, operation, isolationLevel
from (DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders);

In [0]:
%sql
ALTER TABLE my_first_catalog.my_first_schema.store_orders SET TBLPROPERTIES ('delta.isolationLevel' = 'Serializable')

In [0]:
%sql
UPDATE my_first_catalog.my_first_schema.store_orders SET sale_price=100.00 WHERE order_number=500;

In [0]:
%sql
SELECT version, operation, isolationLevel 
from (DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders);

In [0]:
%sql
DESCRIBE HISTORY my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
DESCRIBE DETAIL my_first_catalog.my_first_schema.store_orders

In [0]:
%sql
OPTIMIZE my_first_catalog.my_first_schema.store_orders
ZORDER BY (order_number)

In [0]:
%sql
DESCRIBE DETAIL my_first_catalog.my_first_schema.store_orders

In [0]:
%sql
VACUUM my_first_catalog.my_first_schema.store_orders;

In [0]:
%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM my_first_catalog.my_first_schema.store_orders RETAIN 1 HOURS;